# Blinkit Data Analytics — Data validation
DuckDB SQL + Python | Yash Prajapati

## 1. Setup

### 1.1 Libraries

In [1]:
import warnings, math, textwrap
warnings.filterwarnings('ignore')
import duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')
plt.rcParams.update({'figure.figsize':(10,4.5),'axes.grid':True,'grid.alpha':.25,'axes.spines.top':False,'axes.spines.right':False,'font.size':10})
print('duckdb', duckdb.__version__, '| pandas', pd.__version__, '| numpy', np.__version__)

duckdb 1.5.5 | pandas 3.0.2 | numpy 2.4.4


### 1.2 Load cleaned workbook

In [2]:
XLSX = 'data/Blinkit_analysis_new.xlsx'
book = pd.read_excel(XLSX, sheet_name=None)
geo = pd.read_csv('data/city_state_zone.csv')
for name, df in book.items():
    print(f'{name:28s} {df.shape[0]:>6,} rows  {df.shape[1]:>3} cols')

Data_Quality_Report              46 rows    3 cols
Orders_Customer_Info          5,000 rows   16 cols
Orders_Raw_Archive            5,000 rows   23 cols
Order_Line_Items              5,000 rows   14 cols
Delivery_Performance          5,000 rows    8 cols
Customer_Feedback             5,000 rows    8 cols
Customers                     2,500 rows   11 cols
Products                        268 rows   10 cols
Inventory_Analysis              268 rows   18 cols
Data_Dictionary                  44 rows    4 cols
Reference_Parameters             11 rows    5 cols
Relational_Diagram                0 rows    0 cols
Pivot_Payment_Method              7 rows    7 cols
Pivot_Customer_Segment            7 rows    5 cols
Pivot_Category_Sales             14 rows    4 cols
Pivot_Monthly_Trend              24 rows    6 cols
Pivot_Area_Delivery              23 rows    6 cols
Pivot_Inventory_Movement          6 rows    7 cols
Pivot_Category_Stock             14 rows    8 cols


### 1.3 Create DuckDB database and load tables

In [3]:
con = duckdb.connect('blinkit.duckdb')
load = {'orders_src':'Orders_Raw_Archive','items_src':'Order_Line_Items','delivery_src':'Delivery_Performance',
        'feedback_src':'Customer_Feedback','customers_src':'Customers','products_src':'Products','inventory_src':'Inventory_Analysis'}
for tbl, sheet in load.items():
    df = book[sheet].copy()
    df.columns = [c.strip() for c in df.columns]
    con.register('tmp_df', df)
    con.execute(f'CREATE OR REPLACE TABLE {tbl} AS SELECT * FROM tmp_df')
con.register('geo_df', geo)
con.execute('CREATE OR REPLACE TABLE geo AS SELECT * FROM geo_df')
con.execute("SELECT table_name, estimated_size FROM duckdb_tables() ORDER BY table_name").df()

,table_name,estimated_size
0,customers_src,2500
1,delivery_src,5000
2,feedback_src,5000
3,geo,316
4,inventory_src,268
5,items_src,5000
6,orders_src,5000
7,products_src,268


### 1.4 Typed analysis views

In [4]:
con.execute('''
CREATE OR REPLACE VIEW orders AS
SELECT order_id, customer_id,
       strptime(order_date, '%d-%m-%Y %H:%M')              AS order_ts,
       CAST(strptime(order_date, '%d-%m-%Y %H:%M') AS DATE) AS order_date,
       strptime(promised_delivery_time, '%d-%m-%Y %H:%M')   AS promised_ts,
       strptime(actual_delivery_time,  '%d-%m-%Y %H:%M')    AS actual_ts,
       delivery_status, order_total, payment_method, delivery_partner_id, store_id,
       CAST(delivery_time_minutes AS INTEGER)               AS delay_min,
       distance_km, reasons_if_delayed, customer_name,
       trim(area) AS area, pincode, customer_segment,
       CAST(registration_date AS DATE)                      AS registration_date,
       order_day_of_week, order_time_slot, order_value_segment,
       CASE WHEN is_weekend = 'Yes' THEN 1 ELSE 0 END       AS is_weekend
FROM orders_src ''')

con.execute('''
CREATE OR REPLACE VIEW items AS
SELECT order_id, product_id, quantity, unit_price, product_name, category, brand,
       price, mrp, margin_percentage, shelf_life_days, min_stock_level, max_stock_level, line_total,
       line_total * margin_percentage / 100.0 AS margin_value
FROM items_src ''')

con.execute('''
CREATE OR REPLACE VIEW feedback AS
SELECT feedback_id, order_id, customer_id, rating, feedback_category, sentiment,
       CAST(feedback_date AS DATE) AS feedback_date
FROM feedback_src ''')

con.execute('''
CREATE OR REPLACE VIEW f_sales AS
SELECT o.order_id, o.customer_id, o.order_ts, o.order_date,
       date_trunc('month', o.order_date)  AS order_month,
       extract(hour FROM o.order_ts)      AS order_hour,
       o.order_day_of_week, o.is_weekend, o.order_time_slot, o.order_value_segment,
       o.payment_method, o.customer_segment, o.registration_date, o.customer_name,
       o.area, g.state, g.zone, g.city_tier,
       i.product_id, i.product_name, i.category, i.brand,
       i.quantity, i.line_total AS revenue, i.margin_value, i.margin_percentage,
       i.price, i.mrp, i.shelf_life_days,
       o.delay_min, o.distance_km, o.delivery_status,
       CASE WHEN o.delivery_status = 'On Time' THEN 1 ELSE 0 END AS is_on_time_status,
       CASE WHEN o.delay_min > 0 THEN 1 ELSE 0 END               AS is_late_minutes,
       f.rating, f.sentiment, f.feedback_category
FROM orders o
JOIN items i    ON i.order_id = o.order_id
LEFT JOIN geo g ON g.area     = o.area
LEFT JOIN feedback f ON f.order_id = o.order_id ''')

con.execute('SELECT COUNT(*) AS fact_rows, COUNT(DISTINCT order_id) AS orders, MIN(order_date) AS first_day, MAX(order_date) AS last_day FROM f_sales').df()

,fact_rows,orders,first_day,last_day
0,5000,5000,2023-03-16,2024-11-04


### 1.5 Query helper

In [5]:
def q(sql, con=con):
    return con.execute(textwrap.dedent(sql)).df()

def pct(x, n):
    return round(100.0 * x / n, 2) if n else 0.0

q('SELECT COUNT(*) AS rows_in_fact_view FROM f_sales')

,rows_in_fact_view
0,5000


## 2. Data validation

### 2.1 Row and key counts per table

In [6]:
q('''
SELECT 'orders'    AS table_name, COUNT(*) AS rows, COUNT(DISTINCT order_id)    AS unique_keys FROM orders
UNION ALL SELECT 'order_items', COUNT(*), COUNT(DISTINCT order_id)    FROM items
UNION ALL SELECT 'customers',   COUNT(*), COUNT(DISTINCT customer_id) FROM customers_src
UNION ALL SELECT 'products',    COUNT(*), COUNT(DISTINCT product_id)  FROM products_src
UNION ALL SELECT 'delivery',    COUNT(*), COUNT(DISTINCT order_id)    FROM delivery_src
UNION ALL SELECT 'feedback',    COUNT(*), COUNT(DISTINCT feedback_id) FROM feedback
ORDER BY table_name ''')

,table_name,rows,unique_keys
0,customers,2500,2500
1,delivery,5000,5000
2,feedback,5000,5000
3,order_items,5000,5000
4,orders,5000,5000
5,products,268,268


### 2.2 Duplicate primary keys

In [7]:
q('''
WITH d AS (
  SELECT 'orders.order_id'       AS key_name, order_id::VARCHAR    AS key_value, COUNT(*) AS n FROM orders    GROUP BY 2
  UNION ALL SELECT 'customers.customer_id', customer_id::VARCHAR, COUNT(*) FROM customers_src GROUP BY 2
  UNION ALL SELECT 'products.product_id',   product_id::VARCHAR,  COUNT(*) FROM products_src  GROUP BY 2
  UNION ALL SELECT 'feedback.feedback_id',  feedback_id::VARCHAR, COUNT(*) FROM feedback      GROUP BY 2
)
SELECT key_name, COUNT(*) FILTER (WHERE n > 1) AS duplicate_keys, COUNT(*) AS distinct_keys
FROM d GROUP BY key_name ORDER BY key_name ''')

,key_name,duplicate_keys,distinct_keys
0,customers.customer_id,0,2500
1,feedback.feedback_id,0,5000
2,orders.order_id,0,5000
3,products.product_id,0,268


### 2.3 Missing values by column

In [8]:
frames = {'orders':book['Orders_Raw_Archive'],'order_items':book['Order_Line_Items'],
          'customers':book['Customers'],'products':book['Products'],
          'delivery':book['Delivery_Performance'],'feedback':book['Customer_Feedback']}
rows = []
for name, df in frames.items():
    for col in df.columns:
        miss = int(df[col].isna().sum()) + int((df[col].astype(str).str.strip() == '').sum())
        if miss:
            rows.append({'table':name,'column':col,'missing':miss,'missing_pct':pct(miss,len(df))})
pd.DataFrame(rows).sort_values('missing', ascending=False).reset_index(drop=True)

,table,column,missing,missing_pct
0,orders,reasons_if_delayed,1902,38.04
1,delivery,reasons_if_delayed,1902,38.04


### 2.4 Referential integrity

In [9]:
q('''
SELECT 'items.order_id -> orders'        AS relation, COUNT(*) AS orphan_rows FROM items i    LEFT JOIN orders o ON o.order_id = i.order_id WHERE o.order_id IS NULL
UNION ALL SELECT 'items.product_id -> products', COUNT(*) FROM items i     LEFT JOIN products_src p  ON p.product_id  = i.product_id  WHERE p.product_id  IS NULL
UNION ALL SELECT 'orders.customer_id -> customers', COUNT(*) FROM orders o LEFT JOIN customers_src c ON c.customer_id = o.customer_id WHERE c.customer_id IS NULL
UNION ALL SELECT 'delivery.order_id -> orders', COUNT(*) FROM delivery_src d LEFT JOIN orders o ON o.order_id = d.order_id WHERE o.order_id IS NULL
UNION ALL SELECT 'feedback.order_id -> orders', COUNT(*) FROM feedback f   LEFT JOIN orders o ON o.order_id = f.order_id WHERE o.order_id IS NULL
UNION ALL SELECT 'orders.area -> geo', COUNT(*) FROM orders o LEFT JOIN geo g ON g.area = o.area WHERE g.area IS NULL ''')

,relation,orphan_rows
0,items.order_id -> orders,0
1,items.product_id -> products,0
2,orders.customer_id -> customers,0
3,delivery.order_id -> orders,0
4,feedback.order_id -> orders,0
5,orders.area -> geo,0


### 2.5 Business rule checks

In [10]:
q('''
SELECT 'quantity <= 0'                      AS rule, COUNT(*) AS failing_rows FROM items WHERE quantity <= 0
UNION ALL SELECT 'unit price > MRP',              COUNT(*) FROM items  WHERE price > mrp
UNION ALL SELECT 'line total <> qty * price',     COUNT(*) FROM items  WHERE abs(line_total - quantity * price) > 0.01
UNION ALL SELECT 'delivery before order time',    COUNT(*) FROM orders WHERE actual_ts < order_ts
UNION ALL SELECT 'promised before order time',    COUNT(*) FROM orders WHERE promised_ts < order_ts
UNION ALL SELECT 'rating outside 1-5',            COUNT(*) FROM feedback WHERE rating < 1 OR rating > 5
UNION ALL SELECT 'shelf life <= 0',               COUNT(*) FROM products_src WHERE shelf_life_days <= 0
UNION ALL SELECT 'min stock > max stock',         COUNT(*) FROM products_src WHERE min_stock_level > max_stock_level
UNION ALL SELECT 'registration after first order',COUNT(*) FROM (
      SELECT customer_id, MIN(order_date) AS first_order, MIN(registration_date) AS reg FROM orders GROUP BY 1)
      WHERE reg > first_order ''')

,rule,failing_rows
0,quantity <= 0,0
1,unit price > MRP,0
2,line total <> qty * price,0
3,delivery before order time,0
4,promised before order time,0
5,rating outside 1-5,0
6,shelf life <= 0,0
7,min stock > max stock,0
8,registration after first order,1438


### 2.6 Order total in orders table vs sum of line items

In [11]:
q('''
WITH cmp AS (
  SELECT o.order_id, o.order_total, i.line_total,
         round(o.order_total - i.line_total, 2) AS diff
  FROM orders o JOIN items i ON i.order_id = o.order_id
)
SELECT COUNT(*) AS orders_compared,
       COUNT(*) FILTER (WHERE abs(diff) > 0.01) AS mismatched_orders,
       round(AVG(diff), 2)  AS avg_difference,
       round(SUM(order_total), 2) AS total_in_orders_table,
       round(SUM(line_total), 2)  AS total_in_items_table
FROM cmp ''')

,orders_compared,mismatched_orders,avg_difference,total_in_orders_table,total_in_items_table
0,5000,4999,"1,207.38","11,009,308.50","4,972,415.43"


### 2.7 Items per order

In [12]:
q('''
SELECT items_per_order, COUNT(*) AS orders FROM (
  SELECT order_id, COUNT(*) AS items_per_order FROM items GROUP BY 1)
GROUP BY 1 ORDER BY 1 ''')

,items_per_order,orders
0,1,5000


### 2.8 Customer segment label vs actual ordering behaviour

In [13]:
q('''
SELECT c.customer_segment                              AS label,
       COUNT(DISTINCT c.customer_id)                   AS customers_labelled,
       COUNT(DISTINCT o.customer_id)                   AS customers_who_ordered,
       COUNT(o.order_id)                               AS orders,
       round(SUM(o.order_total), 0)                    AS revenue,
       round(COUNT(o.order_id) * 1.0 / NULLIF(COUNT(DISTINCT o.customer_id), 0), 2) AS orders_per_buyer
FROM customers_src c
LEFT JOIN orders o ON o.customer_id = c.customer_id
GROUP BY 1 ORDER BY revenue DESC ''')

,label,customers_labelled,customers_who_ordered,orders,revenue,orders_per_buyer
0,Regular,639,574,1320,"2,890,149.00",2.30
1,New,628,530,1222,"2,795,854.00",2.31
2,Premium,633,550,1268,"2,731,330.00",2.31
3,Inactive,600,518,1190,"2,591,976.00",2.30


### 2.9 Date coverage

In [14]:
q('''
SELECT MIN(order_date) AS first_order, MAX(order_date) AS last_order,
       date_diff('day', MIN(order_date), MAX(order_date)) AS days_covered,
       COUNT(DISTINCT order_date)  AS days_with_orders,
       COUNT(DISTINCT date_trunc('month', order_date)) AS months_covered
FROM orders ''')

,first_order,last_order,days_covered,days_with_orders,months_covered
0,2023-03-16,2024-11-04,599,600,21
